# Extract training samples
* Download low-tide cloud free satellite iamges closest to the UAV image collection
* Sample the satellite image bands where appromximately a single UAV class

In [ ]:
import pathlib
import numpy
import dask.distributed
import pandas

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

# Values to edit

In [ ]:
site_names_no_data_0_but_not_set = ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai", "Takapuwahia_Nov25"]
site_names_no_data_neg128_and_set = ["IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
all_site_names = ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()

uav_folder_in = data_path / "classified_orthos"
uav_folder_out = data_path / "classified_uav"

uav_class_labels_file = data_path / "ELF24505_ClassificationClasses.txt"

## Files already with no data value set
* Ensure no data values are updated to utils.UAV_NAN_CLASS
* Ensure no data, CRS and transform are written in the metadata

In [ ]:
for site_name in site_names_no_data_neg128_and_set:
    uav_file = uav_folder_in / f"{site_name}_classified.tif"
    uav_data = utils.load_classification(
        filename=uav_file,
        chunks=True, masked=False
    )
    print(f"site {site_name} no data value {uav_data.rio.nodata} crs {uav_data.rio.crs}")
    '''uav_data = uav_data.where(uav_data != uav_data.rio.nodata, utils.UAV_NAN_CLASS)
    uav_data.rio.write_nodata(utils.UAV_NAN_CLASS, inplace=True)
    utils.write_netcdf_conventions_in_place(uav_data)
    utils.save_tiff(uav_data, uav_folder_out / uav_file.name)'''

## Files without nodata metadata - 0 used where nodata
* Ensure no data values are updated to utils.UAV_NAN_CLASS
* Ensure no data, CRS and transform are written in the metadata

In [ ]:
no_data_value = 0
for site_name in site_names_no_data_0_but_not_set:
    uav_file = uav_folder_in / f"{site_name}_classified.tif"
    uav_data = utils.load_classification(
        filename=uav_file,
        chunks=True, masked=False
    )
    print(f"site {site_name} no data value {uav_data.rio.nodata} crs {uav_data.rio.crs}")
    '''uav_data = uav_data.where(uav_data != no_data_value, utils.UAV_NAN_CLASS)
    uav_data.rio.write_nodata(utils.UAV_NAN_CLASS, inplace=True)
    utils.write_netcdf_conventions_in_place(uav_data)
    utils.save_tiff(uav_data, uav_folder_out / uav_file.name)'''

In [ ]:
uav_data

# Show converted files all have the same no data value

In [ ]:
uav_class_labels = utils.read_uav_classe_labels_file(uav_class_labels_file)

for site_name in all_site_names:
    std_uav_file = uav_folder_out / f"{site_name}_classified.tif"
    uav_data_std = utils.load_classification(
        filename=std_uav_file,
        chunks=True, masked=False
    )
    uav_classes_present = numpy.unique(uav_data_std.values)
    label_names = [key for key, value in uav_class_labels.items() if value in uav_classes_present]
    print(f"{site_name} values: {uav_classes_present}, no data is {uav_data_std.rio.nodata}")
    print(f"\tLabels: {label_names}")
    

# Table of UAV pixels per class per site

In [ ]:
uav_class_labels = utils.read_uav_classe_labels_file(uav_class_labels_file)

uav_counts_by_site = {"name": [], "resolution": [], **{key: [] for key in uav_class_labels.keys()}}

for site_name in all_site_names:
    std_uav_file = uav_folder_out / f"{site_name}_classified.tif"
    uav_data_std = utils.load_classification(
        filename=std_uav_file,
        chunks=True, masked=False
    )
    print(f"{site_name} - sum class pixels")
    uav_counts_by_site["name"].append(site_name)
    uav_counts_by_site["resolution"].append(uav_data_std.rio.resolution()[0])
    for key in uav_class_labels.keys():
        #print(f"\t{key}")
        uav_counts_by_site[key].append(
            (uav_data_std.values==uav_class_labels[key]).sum()
        )
uav_counts_by_site = pandas.DataFrame(uav_counts_by_site)

In [ ]:
pandas.DataFrame(uav_counts_by_site)

In [ ]:
uav_counts_by_site = pandas.DataFrame(uav_counts_by_site)
uav_counts_by_site.to_csv(uav_folder_out / "uav_class_pixel_summary.csv", index=False)

In [ ]:
pandas.set_option('display.max_columns', None)

summary = uav_counts_by_site[[col for col in uav_counts_by_site.columns if col != 'name']
].mul(uav_counts_by_site['resolution']**2, axis=0).astype(int).drop(
    columns=['resolution']).set_axis(uav_counts_by_site["name"], axis=0)

summary.loc['Total'] = summary.sum()
summary[['Seagrass', 'Seagrass submerged', 'Gracilaria', 'Gracilaria submerged', 'Ulva', 'Ulva mats', 'Unvegetated', 'Water', 'Terrestrial', 'Submerged vegetation', 'Microphytobenthos',
 'Cystophora', 'Hormosira', 'Rock', 'Filamentous brown algae', 'Saltmarsh', 'Brown algae mixed', 'Green algae mixed', 'Red algae mixed', 'Shadow', 'Glare']]

In [ ]:
summary[['Seagrass', 'Seagrass submerged', 'Gracilaria', 'Gracilaria submerged', 'Ulva', 'Ulva mats', 'Unvegetated', 'Water', 'Terrestrial', 'Submerged vegetation', 'Microphytobenthos',
 'Cystophora', 'Hormosira', 'Rock', 'Filamentous brown algae', 'Saltmarsh', 'Brown algae mixed', 'Green algae mixed', 'Red algae mixed', 'Shadow', 'Glare']].to_csv(uav_folder_out / "uav_class_m2_summary.csv")